In [1]:
import pandas as pd
import os

# -------------------------------------------------------------------
# FULL WINDOWS PATHS TO YOUR DATA
# -------------------------------------------------------------------
base_dir = r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\data\modeling_data"

path_0618_byplanet = os.path.join(base_dir, "all_data_by_planet_online_06_18_24.csv")
path_0702_byplanet = os.path.join(base_dir, "all_data_by_planet_online_07_02_24.csv")

path_0618_all = os.path.join(base_dir, "all_data_online_06_18_24.csv")
path_0702_all = os.path.join(base_dir, "all_data_online_07_02_24.csv")

# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------
df_bp_0618 = pd.read_csv(path_0618_byplanet)
df_bp_0702 = pd.read_csv(path_0702_byplanet)

df_all_0618 = pd.read_csv(path_0618_all)
df_all_0702 = pd.read_csv(path_0702_all)

# -------------------------------------------------------------------
# MERGE USING 'fileID' AS UNIQUE SUBJECT IDENTIFIER
# UNION OF ROWS, KEEPING ALL SUBJECTS
# -------------------------------------------------------------------
id_col = "fileID"

# --- By-planet combined file ---
df_byplanet_final = pd.concat([df_bp_0618, df_bp_0702], axis=0, ignore_index=True)
df_byplanet_final = df_byplanet_final.drop_duplicates()

# --- All-data combined file ---
df_all_final = pd.concat([df_all_0618, df_all_0702], axis=0, ignore_index=True)
df_all_final = df_all_final.drop_duplicates()

# -------------------------------------------------------------------
# COUNT UNIQUE SUBJECTS
# -------------------------------------------------------------------
unique_bp = df_byplanet_final[id_col].nunique()
unique_all = df_all_final[id_col].nunique()

print("Unique subjects in FINAL by-planet file:", unique_bp)
print("Unique subjects in FINAL all-data file:", unique_all)

# -------------------------------------------------------------------
# SAVE OUTPUT
# -------------------------------------------------------------------
output_bp = os.path.join(base_dir, "all_data_by_planet_online_final.csv")
output_all = os.path.join(base_dir, "all_data_online_final.csv")

df_byplanet_final.to_csv(output_bp, index=False)
df_all_final.to_csv(output_all, index=False)

print("\nSaved combined files:")
print(" -", output_bp)
print(" -", output_all)


Unique subjects in FINAL by-planet file: 297
Unique subjects in FINAL all-data file: 345

Saved combined files:
 - C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\data\modeling_data\all_data_by_planet_online_final.csv
 - C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\data\modeling_data\all_data_online_final.csv


In [4]:
pip install h5py

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 55.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import re
import glob
import h5py
import pandas as pd

# -------------------------------------------------------------------
# Directory containing your JLD parameter files
# -------------------------------------------------------------------
data_dir = r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\model\fit_params\adaptive_discount_online_07_02_24"

# Pattern for all subject JLD files
pattern = os.path.join(data_dir, "sub*.jld")

rows = []

for filepath in sorted(glob.glob(pattern)):
    fname = os.path.basename(filepath)      # e.g., "sub32.jld"
    fileID = os.path.splitext(fname)[0]     # e.g., "sub32"

    # try to extract numeric subject index from filename (optional)
    m = re.search(r"\d+", fileID)
    subj_num = int(m.group()) if m else None

    with h5py.File(filepath, "r") as f:
        # "res" is a scalar record with fields a_, b_, c_
        res = f["res"][()]   # numpy.void with dtype names ('a_', 'b_', 'c_')
        # get field names from dtype
        field_names = res.dtype.names

        row = {
            "fileID": fileID,
            "sub_num": subj_num,
        }

        # add each parameter field to row
        for name in field_names:
            row[name] = float(res[name])

        rows.append(row)

# -------------------------------------------------------------------
# Build DataFrame and save to CSV
# -------------------------------------------------------------------
df = pd.DataFrame(rows)

out_path = os.path.join(data_dir, "adaptive_discount_online_07_02_24_params.csv")
df.to_csv(out_path, index=False)

print("Saved parameter CSV to:")
print(out_path)
print("\nPreview:")
print(df.head())




Saved parameter CSV to:
C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\model\fit_params\adaptive_discount_online_07_02_24\adaptive_discount_online_07_02_24_params.csv

Preview:
   fileID  sub_num        a_        b_        c_
0    sub1        1  5.019531 -1.976562  2.449219
1   sub10       10  0.039062  9.953125  1.617188
2  sub100      100  1.562500 -0.125000  0.187500
3  sub101      101  9.550781  2.335938  0.105469
4  sub102      102  9.980469  4.070312 -2.074219
